# Entrenamiento YOLOv8-Pose – Giroscopio Dron

Este notebook entrena un modelo YOLOv8s-pose personalizado para detectar
los 9 keypoints del giroscopio y calcular Roll, Pitch y Yaw.

**Pasos:**
1. Instalar dependencias
2. Descargar dataset desde Roboflow
3. Entrenar YOLOv8s-pose
4. Evaluar resultados
5. Exportar a ONNX para compilar a HEF (Hailo)

> ⚠️ Asegúrate de tener activada la GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Instalar dependencias

In [ ]:
!pip install ultralytics roboflow --quiet
import ultralytics
ultralytics.checks()

## 3. Descargar dataset desde Roboflow

Pega aquí el snippet que te dio Roboflow (sustituye el bloque de abajo)

In [ ]:
# ─── PEGA AQUÍ TU SNIPPET DE ROBOFLOW ───────────────────────────────────────
# Ejemplo (sustituye con el tuyo):
#
# !pip install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="TU_API_KEY")
# project = rf.workspace("joss-workspace-7orjh").project("giroscopio-dron")
# version = project.version(1)
# dataset = version.download("yolov8")
# ─────────────────────────────────────────────────────────────────────────────

# Guardar la ruta del dataset
DATASET_PATH = dataset.location
print(f'Dataset en: {DATASET_PATH}')

## 4. Revisar estructura del dataset

In [ ]:
import os, yaml

# Mostrar estructura
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')

# Leer data.yaml
yaml_path = os.path.join(DATASET_PATH, 'data.yaml')
with open(yaml_path) as f:
    data = yaml.safe_load(f)
print('\nContenido data.yaml:')
print(data)

## 5. Entrenar YOLOv8s-pose

- `epochs=100` — suficiente para un dataset de ~400 imágenes
- `imgsz=640` — coincide con el preprocesado de Roboflow
- `batch=16` — ajustar si hay OOM (reducir a 8)
- `patience=20` — early stopping si no mejora en 20 épocas

In [ ]:
from ultralytics import YOLO

# Cargar modelo base YOLOv8s-pose (preentrenado en COCO)
model = YOLO('yolov8s-pose.pt')

# Entrenar
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,              # GPU
    project='giroscopio',
    name='yolov8s_pose_v1',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    augment=True,
    degrees=10.0,          # rotación ±10° adicional
    scale=0.3,             # escala ±30%
    fliplr=0.0,            # NO flip horizontal (izq/der son distintos)
    mosaic=0.5,
)

print('\nEntrenamiento completado!')
print(f'Mejor modelo: {results.save_dir}/weights/best.pt')

## 6. Evaluar el modelo

In [ ]:
# Cargar mejor modelo y evaluar en test set
best_model = YOLO(f'giroscopio/yolov8s_pose_v1/weights/best.pt')
metrics = best_model.val(data=yaml_path, split='test')

print(f'\nmAP50:     {metrics.pose.map50:.3f}')
print(f'mAP50-95:  {metrics.pose.map:.3f}')

## 7. Visualizar predicciones en imágenes de test

In [ ]:
import glob
from IPython.display import Image, display

# Inferencia en algunas imágenes de test
test_images = glob.glob(os.path.join(DATASET_PATH, 'test/images/*.jpg'))[:6]

results = best_model.predict(
    source=test_images,
    save=True,
    project='giroscopio_test',
    name='predicciones',
    conf=0.3,
)

# Mostrar resultados
pred_imgs = glob.glob('giroscopio_test/predicciones/*.jpg')
for img_path in pred_imgs[:6]:
    display(Image(img_path, width=400))

## 8. Exportar a ONNX

El modelo ONNX es el formato intermedio necesario para compilarlo
a HEF con el Hailo Dataflow Compiler.

In [ ]:
# Exportar a ONNX con opset 11 (requerido por Hailo)
best_model.export(
    format='onnx',
    opset=11,
    simplify=True,
    imgsz=640,
)

onnx_path = 'giroscopio/yolov8s_pose_v1/weights/best.onnx'
print(f'Modelo ONNX exportado: {onnx_path}')
print(f'Tamaño: {os.path.getsize(onnx_path)/1024/1024:.1f} MB')

## 9. Descargar archivos resultantes

Descarga estos dos archivos para compilar el HEF en Hailo:

In [ ]:
from google.colab import files
import shutil

# Comprimir y descargar
shutil.copy(
    'giroscopio/yolov8s_pose_v1/weights/best.pt',
    '/content/giroscopio_pose_best.pt'
)
shutil.copy(
    'giroscopio/yolov8s_pose_v1/weights/best.onnx',
    '/content/giroscopio_pose_best.onnx'
)

print('Descargando best.pt ...')
files.download('/content/giroscopio_pose_best.pt')

print('Descargando best.onnx ...')
files.download('/content/giroscopio_pose_best.onnx')

print('\n✅ Listo. Guarda estos dos archivos, los necesitarás para compilar el HEF.')

## Próximos pasos

Con el archivo `best.onnx` descargado, el siguiente paso es compilarlo
a formato `.hef` usando el **Hailo Dataflow Compiler** (se hace en PC, no en Raspberry).
El proceso se detalla en la documentación de Hailo:
https://hailo.ai/developer-zone/documentation/

Una vez tengas el `.hef`, simplemente sustituye la ruta en `script_giroscopio_ai.py`:
```python
--hef /ruta/a/giroscopio_pose.hef
```